# Phase 4 — Court Judgment Prediction (CJPE) · Kaggle 2× T4

**Inputs:** `prefixlm-run1` (tokenizer) + `phase1-models` (your four Phase-1 models).
Settings: GPU T4 ×2, Internet On, secret `HF_TOKEN` (tick it in Add-ons → Secrets).

**Yours (rule 17.5):** `cj_model.py` (chunk → case representation, head, adaptation methods incl. LoRMA, KD student) and every `TODO`.
**Harness:** data, tokenization cache, chunking (first / last / head_tail / spread / all), training loop, dev macro-F1,
class weights, optional dev-tuned threshold, KD plumbing, `prediction.json` = `[{"id", "pred"}]` + zip.

Phase-4 technique list (doc 10.2.1): Full FT · Partial/layer-wise FT · Layer freezing/unfreezing · BitFit · **LoRMA** · AdaLoRA · Adapters · **KD**.
(LoRA, rsLoRA, prompt and prefix tuning are NOT on this list.)

Order: cells 1-6 once (tests pass, data check, SMOKE TEST PASSED) → write `cj_model.py` → model check → experiments.

In [ ]:
# 1 — setup
import os, json, glob, shutil, time, subprocess, sys
os.chdir("/kaggle/working")
import torch
NGPU = torch.cuda.device_count(); print("GPUs:", NGPU)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret not available:", e)

def find(pattern):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

In [ ]:
# 2 — Phase-4 data + tokenizer
from huggingface_hub import hf_hub_download
REPO = "Exploration-Lab/CS-781-Capstone"
P4 = {}
for f in ["Train/train.json", "Train/dev.json", "Test/test.json"]:
    P4[f.split("/")[-1].replace(".json", "")] = hf_hub_download(REPO, f"Phase_4_CJPE/{f}", repo_type="dataset")
print(json.dumps(P4, indent=1))
os.makedirs("tokenizer", exist_ok=True)
for fn in ["vocab.json", "merges.txt"]:
    shutil.copy(find(fn)[0], f"tokenizer/{fn}")

In [ ]:
%%writefile cj_data.py
"""cj_data.py — Phase-4 Court Judgment Prediction data handling."""
import hashlib, json, os, pickle
from collections import Counter

import numpy as np

ID_KEYS = ("id", "case_id", "uid", "doc_id", "name")
TEXT_KEYS = ("text", "document", "doc", "judgment", "judgement", "case_text", "content")
LABEL_KEYS = ("label", "labels", "outcome", "decision", "y", "target")


def _get(rec, keys):
    for k in keys:
        if k in rec:
            return rec[k]
    return None


def _norm_label(v):
    if v is None:
        return None
    if isinstance(v, (list, tuple)):
        assert len(v) == 1, f"expected one label, got {v}"
        v = v[0]
    v = int(float(v))
    assert v in (0, 1), f"label {v} is not 0/1"
    return v


def load_cases(path):
    obj = json.load(open(path, encoding="utf-8"))
    if isinstance(obj, dict) and not any(k in obj for k in ID_KEYS):
        for k in ("data", "cases", "examples", "instances"):
            if isinstance(obj.get(k), list):
                obj = obj[k]
                break
    recs = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, str):
                recs.append(dict(id=k, text=v, label=None))
            else:
                r = dict(v); r.setdefault("id", k); recs.append(r)
    else:
        recs = list(obj)
    out = {}
    for i, r in enumerate(recs):
        cid = _get(r, ID_KEYS)
        cid = i if cid is None else cid
        text = _get(r, TEXT_KEYS)
        assert text is not None, f"record {cid}: no text field among {TEXT_KEYS}; keys {list(r)[:8]}"
        if isinstance(text, (list, tuple)):
            text = "\n".join(str(t) for t in text)
        key, text, lab = str(cid), str(text), _norm_label(_get(r, LABEL_KEYS))
        if key in out:
            if out[key]["text"] == text and out[key]["label"] == lab:
                DUPS["identical_dropped"] += 1
                continue
            j = 2
            while f"{key}#{j}" in out:
                j += 1
            key = f"{key}#{j}"
            DUPS["different_kept"] += 1
        out[key] = dict(id=cid, text=text, label=lab)
    return out


DUPS = Counter()


def tokenize_cases(cases, encode_batch, cache_dir=None, tag=""):
    h = hashlib.md5((tag + "|".join(f"{k}:{len(v['text'])}" for k, v in cases.items())).encode()).hexdigest()[:16]
    path = os.path.join(cache_dir, f"cj_tok_{h}.pkl") if cache_dir else None
    if path and os.path.exists(path):
        return pickle.load(open(path, "rb"))
    keys = list(cases)
    out = {}
    for i in range(0, len(keys), 512):
        ks = keys[i:i + 512]
        for k, ids in zip(ks, encode_batch([cases[k]["text"] for k in ks])):
            out[k] = np.asarray(ids, dtype=np.uint16)
    if path:
        os.makedirs(cache_dir, exist_ok=True)
        tmp = path + f".tmp{os.getpid()}"
        pickle.dump(out, open(tmp, "wb"), protocol=4)
        os.replace(tmp, path)
    return out


def chunk_doc(n_tokens, chunk_len, stride=None, max_chunks=None, select="first"):
    stride = int(stride or chunk_len)
    assert 0 < stride <= chunk_len, "need 0 < stride <= chunk_len"
    if n_tokens <= chunk_len:
        spans = [(0, max(1, n_tokens))]
    else:
        starts = list(range(0, n_tokens - chunk_len + 1, stride))
        if starts[-1] + chunk_len < n_tokens:
            starts.append(n_tokens - chunk_len)
        spans = [(s, s + chunk_len) for s in starts]
    k = len(spans) if not max_chunks else min(int(max_chunks), len(spans))
    if select == "all" or k == len(spans):
        return spans
    if select == "first":
        return spans[:k]
    if select == "last":
        return spans[-k:]
    if select == "head_tail":
        h = (k + 1) // 2
        return spans[:h] + spans[len(spans) - (k - h):] if k - h else spans[:h]
    if select == "spread":
        idx = sorted({round(i * (len(spans) - 1) / max(1, k - 1)) for i in range(k)}) if k > 1 else [0]
        return [spans[i] for i in idx]
    raise ValueError(f"select must be first | last | head_tail | spread | all, got {select!r}")


def binary_scores(pred, gold):
    keys = list(gold)
    tp, fp, fn = Counter(), Counter(), Counter()
    conf = [[0, 0], [0, 0]]
    for k in keys:
        p, g = int(pred[k]), int(gold[k])
        conf[g][p] += 1
        if p == g:
            tp[p] += 1
        else:
            fp[p] += 1; fn[g] += 1
    per = {}
    for c in (0, 1):
        P = tp[c] / (tp[c] + fp[c]) if tp[c] + fp[c] else 0.0
        R = tp[c] / (tp[c] + fn[c]) if tp[c] + fn[c] else 0.0
        per[c] = dict(p=P, r=R, f1=2 * P * R / (P + R) if P + R else 0.0, support=tp[c] + fn[c], predicted=tp[c] + fp[c])
    m = lambda key: (per[0][key] + per[1][key]) / 2
    return dict(macro_f1=m("f1"), macro_p=m("p"), macro_r=m("r"), accuracy=(tp[0] + tp[1]) / max(1, len(keys)),
                per_class=per, confusion=conf, n=len(keys))


def format_scores(sc, title=""):
    c = sc["confusion"]
    s = (f"{title}macro-F1 {sc['macro_f1']:.4f} | macro-P {sc['macro_p']:.4f} | macro-R {sc['macro_r']:.4f} | "
         f"acc {sc['accuracy']:.4f} | n {sc['n']}\n")
    for k in (0, 1):
        v = sc["per_class"][k]
        s += f"  class {k}: F1 {v['f1']:.3f}  P {v['p']:.3f}  R {v['r']:.3f}  (support {v['support']}, predicted {v['predicted']})\n"
    return s + f"  confusion [gold][pred]: {c}"


def best_threshold(p1, gold):
    keys = list(gold)
    cands = sorted(set(round(float(p1[k]), 4) for k in keys))
    best_t, best_f = 0.5, -1.0
    for t in [0.5] + cands:
        f = binary_scores({k: int(p1[k] >= t) for k in keys}, gold)["macro_f1"]
        if f > best_f + 1e-12:
            best_t, best_f = t, f
    return best_t, best_f


def validate_prediction(pred_list, test_cases):
    assert isinstance(pred_list, list), "prediction.json must be a JSON list"
    ids = [r["id"] for r in pred_list]
    assert len(ids) == len(set(map(str, ids))), "duplicate ids in prediction"
    want = {str(v["id"]) for v in test_cases.values()}
    got = set(map(str, ids))
    assert got == want, f"ids differ from test.json: missing {len(want - got)}, extra {len(got - want)}"
    for r in pred_list:
        assert set(r) == {"id", "pred"}, f"each entry must be exactly {{id, pred}}, got {sorted(r)}"
        assert type(r["pred"]) is int and r["pred"] in (0, 1), f"pred must be int 0/1, got {r['pred']!r}"
    return len(pred_list)


def dataset_stats(cases, toks):
    n = sorted(len(toks[k]) for k in cases)
    q = lambda f: n[min(len(n) - 1, int(len(n) * f))] if n else 0
    lab = Counter(v["label"] for v in cases.values())
    return dict(cases=len(cases), labels={str(k): c for k, c in sorted(lab.items(), key=lambda kv: str(kv[0]))},
                tokens_p10=q(.1), tokens_p50=q(.5), tokens_p90=q(.9), tokens_max=n[-1] if n else 0,
                tokens_total=int(sum(n)))

In [ ]:
%%writefile test_cj_data.py
"""Synthetic tests for cj_data.py (python test_cj_data.py)."""
import json, random, tempfile

import cj_data as cd

ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name} {detail}")


random.seed(0)

for n in [0, 1, 50, 512, 513, 1000, 5000, 73001]:
    for L, S in [(512, 512), (512, 384), (256, 128)]:
        sp = cd.chunk_doc(n, L, S, None, "all")
        cov = set()
        for a, b in sp:
            cov.update(range(a, b))
        good = all(0 <= a < b <= max(1, n) and b - a <= L for a, b in sp)
        full = n == 0 or cov == set(range(n))
        check(f"1 n={n} L={L} S={S}: spans valid + whole doc covered", good and full, f"({len(sp)} chunks)")


sp = cd.chunk_doc(10000, 512, 512, None, "all")
for sel in ("first", "last", "head_tail", "spread"):
    got = cd.chunk_doc(10000, 512, 512, 4, sel)
    check(f"2 select {sel}: 4 chunks, subset, ordered", len(got) == 4 and all(g in sp for g in got) and got == sorted(got))
check("2 first/last are ends", cd.chunk_doc(10000, 512, 512, 3, "first") == sp[:3] and cd.chunk_doc(10000, 512, 512, 3, "last") == sp[-3:])
check("2 head_tail splits", cd.chunk_doc(10000, 512, 512, 3, "head_tail") == sp[:2] + sp[-1:])
check("2 short doc unaffected", cd.chunk_doc(300, 512, 512, 4, "last") == [(0, 300)])


gold = {"a": 1, "b": 1, "c": 0, "d": 0}
pred = {"a": 1, "b": 0, "c": 0, "d": 0}
sc = cd.binary_scores(pred, gold)
f0, f1 = 2 * (2 / 3) * 1 / (2 / 3 + 1), 2 * 1 * .5 / 1.5
check("3 macro-F1", abs(sc["macro_f1"] - (f0 + f1) / 2) < 1e-9 and sc["accuracy"] == .75 and sc["confusion"] == [[2, 0], [1, 1]])
t, f = cd.best_threshold({"a": .9, "b": .4, "c": .3, "d": .1}, gold)
check("3 threshold search finds perfect split", f == 1.0 and .3 < t <= .4, f"(t={t})")


tmp = tempfile.mkdtemp()
rows = [dict(id=f"case_{i}", text="x " * i, label=i % 2) for i in range(10)]
json.dump(rows, open(f"{tmp}/a.json", "w"))
json.dump({r["id"]: dict(text=r["text"], label=str(r["label"])) for r in rows}, open(f"{tmp}/b.json", "w"))
json.dump({"data": [dict(case_id=r["id"], text=[r["text"]], label=[r["label"]]) for r in rows]}, open(f"{tmp}/c.json", "w"))
json.dump([dict(id=r["id"], text=r["text"]) for r in rows], open(f"{tmp}/t.json", "w"))
a, b, c, t = (cd.load_cases(f"{tmp}/{x}.json") for x in "abct")
same = lambda u, v: all(u[k]["label"] == v[k]["label"] and u[k]["text"].strip() == v[k]["text"].strip() for k in u)
check("4 loaders agree", set(a) == set(b) == set(c) and same(a, b) and same(a, c))
check("4 test file has no labels", all(v["label"] is None for v in t.values()))
json.dump([dict(id=7, text="a", label=1)], open(f"{tmp}/n.json", "w"))
check("4 integer id kept as int", cd.load_cases(f"{tmp}/n.json")["7"]["id"] == 7)

json.dump([dict(id="x", text="a", label=1), dict(id="x", text="a", label=1), dict(id="x", text="b", label=0)],
          open(f"{tmp}/d.json", "w"))
dd = cd.load_cases(f"{tmp}/d.json")
check("4 repeated ids: identical dropped, different kept", sorted(dd) == ["x", "x#2"] and dd["x#2"]["label"] == 0)


pl = [dict(id=v["id"], pred=i % 2) for i, v in enumerate(t.values())]
check("5 validator accepts", cd.validate_prediction(pl, t) == 10)
for bad, why in [(pl[:-1], "missing id"), (pl + [pl[0]], "duplicate"), ([dict(r, pred=float(r["pred"])) for r in pl], "float pred"),
                 ([dict(r, extra=1) for r in pl], "extra key")]:
    try:
        cd.validate_prediction(bad, t); check(f"5 validator rejects {why}", False)
    except AssertionError:
        check(f"5 validator rejects {why}", True)

print("\nALL TESTS PASSED" if ok else "\nSOME TESTS FAILED")

In [ ]:
%%writefile cj_train.py
"""cj_train.py — Phase-4 Court Judgment Prediction harness."""
import importlib, json, math, os, random, sys, time, zipfile
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F

import cj_data as cd

CFG = json.load(open(sys.argv[1]))
D, TR, MODEL_CFG = CFG["data"], CFG["train"], CFG["model"]
KD = CFG.get("kd") or {}
OUT, NAME = CFG["out_dir"], CFG["run_name"]
os.makedirs(OUT, exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = int(TR.get("seed", 0))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
C = 2


def log(*a):
    print(*a, flush=True)


# ------------------------------------------------------------------ data
from tokenizers import ByteLevelBPETokenizer
TOK = ByteLevelBPETokenizer(vocab=os.path.join(D["tokenizer_dir"], "vocab.json"),
                            merges=os.path.join(D["tokenizer_dir"], "merges.txt"))
PAD_ID = TOK.token_to_id("<pad>") or 0
L, STRIDE = int(D["chunk_len"]), int(D.get("stride") or D["chunk_len"])
MAXC, SEL = D.get("max_chunks"), D.get("select", "first")
EMAXC, ESEL = D.get("eval_max_chunks", MAXC), D.get("eval_select", SEL)


def enc_batch(texts):
    return [e.ids for e in TOK.encode_batch(texts)]


PARTS, TOKS = {}, {}
for split in ("train", "dev", "test"):
    if D.get(split):
        raw = cd.load_cases(D[split])
        t = cd.tokenize_cases(raw, enc_batch, D.get("cache_dir", "cj_cache"), tag=split)
        PARTS[split] = {f"{split}::{k}": v for k, v in raw.items()}
        TOKS.update({f"{split}::{k}": v for k, v in t.items()})
TRAIN, DEVS, TEST = PARTS.get("train", {}), PARTS.get("dev", {}), PARTS.get("test", {})
if D.get("train_on_dev"):
    TRAIN = dict(TRAIN, **DEVS)
    DEVS = {}
assert TRAIN and all(v["label"] is not None for v in TRAIN.values()), "train missing or has cases without a label"
tr_lab = Counter(v["label"] for v in TRAIN.values())
log(f"train {len(TRAIN)} cases {dict(tr_lab)} | dev {len(DEVS)} | test {len(TEST)} | chunk_len {L} stride {STRIDE} "
    f"| train max_chunks {MAXC} select {SEL} | eval max_chunks {EMAXC} select {ESEL}")


def spans(key, train):
    n = len(TOKS[key])
    return cd.chunk_doc(n, L, STRIDE, MAXC if train else EMAXC, SEL if train else ESEL)


TEACH = None
if KD.get("teacher_logits"):
    z = np.load(KD["teacher_logits"])
    TEACH = {k[2:]: z[k].astype(np.float32) for k in z.files if k.startswith("d:")}
    covered = {k: v for k, v in TRAIN.items() if k in TEACH}
    assert covered, "teacher logits cover none of the train cases (different data or split names?)"
    if len(covered) < len(TRAIN):
        log(f"KD: teacher covers {len(covered)} of {len(TRAIN)} train cases -> the student trains on those {len(covered)}")
        TRAIN = covered
        tr_lab = Counter(v["label"] for v in TRAIN.values())
    log(f"KD: teacher logits for {len(TEACH)} cases | T={KD['temperature']} alpha={KD['alpha']}")


def collate(keys, part, train):
    rows, did, cpos, crel = [], [], [], []
    for j, k in enumerate(keys):
        ids = TOKS[k]
        n = max(1, len(ids))
        for p, (a, b) in enumerate(spans(k, train)):
            seg = ids[a:b] if len(ids) else np.asarray([PAD_ID], np.uint16)
            rows.append(seg); did.append(j); cpos.append(p); crel.append(a / n)
    T = max(len(r) for r in rows)
    x = torch.full((len(rows), T), PAD_ID, dtype=torch.long)
    m = torch.zeros((len(rows), T), dtype=torch.bool)
    for i, r in enumerate(rows):
        x[i, :len(r)] = torch.from_numpy(r.astype(np.int64)); m[i, :len(r)] = True
    y = torch.tensor([part[k]["label"] if part[k]["label"] is not None else -100 for k in keys], dtype=torch.long)
    tch = torch.tensor(np.stack([TEACH[k] for k in keys])) if (train and TEACH is not None) else None
    return (x, m, torch.tensor(did), len(keys), torch.tensor(cpos), torch.tensor(crel, dtype=torch.float32)), y, tch


# ------------------------------------------------------------------ model
model_lib = importlib.import_module(CFG.get("model_module", "cj_model"))
model = model_lib.build_cj_model(dict(MODEL_CFG), C).to(DEV)
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"parameters: total {n_total/1e6:.2f}M | trainable {n_train/1e6:.3f}M ({100*n_train/max(1,n_total):.2f}%)")
assert n_train > 0, "no trainable parameters"

if CFG.get("load_trained"):
    sd = torch.load(CFG["load_trained"], map_location="cpu", weights_only=False)["trained"]
    cur = dict(model.named_parameters())
    miss = [k for k in sd if k not in cur]
    assert not miss, f"load_trained: names not in this model, e.g. {miss[:3]}"
    with torch.no_grad():
        for k, v in sd.items():
            cur[k].copy_(v.to(cur[k].device, cur[k].dtype))
    log(f"loaded {len(sd)} trained tensors from {CFG['load_trained']}")


def run_model(inp):
    x, m, did, nd, cpos, crel = inp
    with torch.autocast(DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
        lg = model(x.to(DEV), m.to(DEV), did.to(DEV), nd, chunk_pos=cpos.to(DEV), chunk_rel=crel.to(DEV))
    assert tuple(lg.shape) == (nd, C), f"model returned {tuple(lg.shape)}, expected [{nd}, {C}]"
    return lg.float()


@torch.no_grad()
def predict_logits(part):
    model.eval()
    keys = sorted(part, key=lambda k: -len(spans(k, False)))
    budget = int(TR.get("eval_chunk_budget", max(1, int(TR["batch_size"])) * int(EMAXC or MAXC or 8)))
    out, i = {}, 0
    while i < len(keys):
        b, used = [], 0
        while i < len(keys) and (not b or used + len(spans(keys[i], False)) <= budget):
            used += len(spans(keys[i], False)); b.append(keys[i]); i += 1
        inp, _, _ = collate(b, part, train=False)
        lg = run_model(inp).cpu().numpy()
        for k, v in zip(b, lg):
            out[k] = v
    model.train()
    return out


THRESH = None


def preds_from(lg):
    if THRESH is None:
        return {k: int(v.argmax()) for k, v in lg.items()}
    return {k: int(torch.softmax(torch.from_numpy(v), -1)[1].item() >= THRESH) for k, v in lg.items()}


def dev_score(lg):
    pred = preds_from(lg)
    return cd.binary_scores(pred, {k: v["label"] for k, v in DEVS.items()}), pred


# ------------------------------------------------------------------ training
bs, acc = int(TR["batch_size"]), int(TR.get("grad_accum", 1))
epochs = int(TR["epochs"])
keys_all = list(TRAIN)
per_epoch = min(len(keys_all), int(TR.get("epoch_cases") or len(keys_all)))
steps_per_epoch = math.ceil(per_epoch / (bs * acc))
total_steps = steps_per_epoch * epochs
warm = int(float(TR.get("warmup_frac", 0)) * total_steps)
base_lr = float(TR["lr"])
groups = model.param_groups(base_lr) if hasattr(model, "param_groups") else \
    [{"params": [p for p in model.parameters() if p.requires_grad]}]
opt = torch.optim.AdamW(groups, lr=base_lr, weight_decay=float(TR.get("weight_decay", 0.0)))
for g in opt.param_groups:
    g.setdefault("initial_lr", g["lr"])
scaler = torch.amp.GradScaler("cuda", enabled=DEV.type == "cuda")
cw = TR.get("class_weights")
if cw == "balanced":
    cw = [len(keys_all) / (2 * max(1, tr_lab[c])) for c in (0, 1)]
cw_t = torch.tensor(cw, dtype=torch.float32, device=DEV) if cw is not None else None
log(f"class weights {cw} | cases/epoch {per_epoch} | steps/epoch {steps_per_epoch} | epochs {epochs}")


def lr_mult(step):
    if step < warm:
        return (step + 1) / max(1, warm)
    t = (step - warm) / max(1, total_steps - warm)
    sch = TR.get("schedule", "linear")
    return 1.0 if sch == "constant" else max(0.0, 1 - t) if sch == "linear" else 0.5 * (1 + math.cos(math.pi * t))


best, best_state, history = -1.0, None, []
if epochs == 0:
    best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
step, t0 = 0, time.time()
max_min = float(TR.get("max_minutes", 1e9))
if DEV.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
order = []
model.train()
for ep in range(1, epochs + 1):
    if len(order) < per_epoch:
        rest = keys_all[:]; random.shuffle(rest); order += rest
    ep_keys, order = order[:per_epoch], order[per_epoch:]
    run_loss, nb = 0.0, 0
    for i in range(0, len(ep_keys), bs):
        inp, y, tch = collate(ep_keys[i:i + bs], TRAIN, train=True)
        lg = run_model(inp)
        y = y.to(DEV)
        loss = F.cross_entropy(lg, y, weight=cw_t)
        if tch is not None:
            Tk, a = float(KD["temperature"]), float(KD["alpha"])
            kl = F.kl_div(F.log_softmax(lg / Tk, -1), F.softmax(tch.to(DEV) / Tk, -1), reduction="batchmean")
            loss = (1 - a) * loss + a * Tk * Tk * kl
        scaler.scale(loss / acc).backward()
        nb += 1; run_loss += loss.item()
        if nb % acc == 0:
            for g in opt.param_groups:
                g["lr"] = g["initial_lr"] * lr_mult(step)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], float(TR.get("grad_clip", 1.0)))
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        if (time.time() - t0) / 60 > max_min:
            break
    rec = dict(epoch=ep, train_loss=run_loss / max(1, nb), minutes=(time.time() - t0) / 60)
    if DEVS:
        sc, _ = dev_score(predict_logits(DEVS))
        rec.update(dev_macro_f1=sc["macro_f1"], dev_acc=sc["accuracy"],
                   dev_pred1=sc["per_class"][1]["predicted"] / max(1, sc["n"]))
        score = sc["macro_f1"]
    else:
        score = ep
    history.append(rec)
    log(" | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
    if score > best:
        best = score
        best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
        log(f"  best so far (epoch {ep})")
    if (time.time() - t0) / 60 > max_min:
        log(f"time limit {max_min} min reached"); break
train_min = (time.time() - t0) / 60
peak_gb = torch.cuda.max_memory_allocated() / 1e9 if DEV.type == "cuda" else 0.0

cur = dict(model.named_parameters())
with torch.no_grad():
    for k, v in best_state.items():
        cur[k].copy_(v.to(DEV))
torch.save(dict(trained=best_state, cfg=CFG), os.path.join(OUT, f"{NAME}_trained_params.pt"))

result = dict(run=NAME, chunk_len=L, stride=STRIDE, max_chunks=MAXC, select=SEL, eval_max_chunks=EMAXC, eval_select=ESEL,
              params_total=n_total, params_trainable=n_train, train_minutes=train_min, peak_mem_gb=peak_gb,
              class_weights=cw, history=history)
if DEVS:
    t1 = time.time()
    dlg = predict_logits(DEVS)
    result["dev_infer_seconds"] = time.time() - t1
    sc, _ = dev_score(dlg)
    result["dev_argmax"] = sc
    log(cd.format_scores(sc, "BEST on dev (argmax) "))
    if TR.get("threshold") == "dev_tuned":
        p1 = {k: float(torch.softmax(torch.from_numpy(v), -1)[1]) for k, v in dlg.items()}
        THRESH, f = cd.best_threshold(p1, {k: v["label"] for k, v in DEVS.items()})
        sc, _ = dev_score(dlg)
        log(cd.format_scores(sc, f"BEST on dev (threshold P(1) >= {THRESH:.4f}, tuned on dev) "))
    result.update(dev=sc, threshold=THRESH)
    json.dump([dict(id=DEVS[k]["id"], pred=v, gold=DEVS[k]["label"]) for k, v in preds_from(dlg).items()],
              open(os.path.join(OUT, f"{NAME}_dev_prediction.json"), "w"))
    np.savez_compressed(os.path.join(OUT, f"{NAME}_dev_logits.npz"), **{f"d:{k}": v for k, v in dlg.items()})

if CFG.get("dump_teacher_logits"):
    tr_keys = sorted(PARTS.get("train", {}))
    if CFG.get("teacher_cases"):
        tr_keys = random.Random(SEED).sample(tr_keys, min(len(tr_keys), int(CFG["teacher_cases"])))
    every = {k: PARTS["train"][k] for k in tr_keys}; every.update(PARTS.get("dev", {}))
    t1 = time.time()
    tl = predict_logits(every)
    np.savez_compressed(os.path.join(OUT, f"{NAME}_teacher_logits.npz"), **{f"d:{k}": v for k, v in tl.items()})
    log(f"teacher logits written for {len(tl)} cases ({(time.time() - t1) / 60:.1f} min)")

if TEST:
    t1 = time.time()
    tlg = predict_logits(TEST)
    np.savez_compressed(os.path.join(OUT, f"{NAME}_test_logits.npz"), **{f"d:{k}": v for k, v in tlg.items()})
    pk = preds_from(tlg)
    pred = [dict(id=TEST[k]["id"], pred=int(pk[k])) for k in TEST]
    n = cd.validate_prediction(pred, TEST)
    pj = os.path.join(OUT, f"{NAME}_prediction.json")
    json.dump(pred, open(pj, "w", encoding="utf-8"), ensure_ascii=False)
    zp = os.path.join(OUT, f"{NAME}_submission.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pj, arcname="prediction.json")
    result.update(test_cases=n, test_infer_seconds=time.time() - t1, test_pred_counts=dict(Counter(r["pred"] for r in pred)))
    log(f"test: {n} cases | predicted {dict(Counter(r['pred'] for r in pred))} | infer {time.time() - t1:.0f}s")
    log(f"wrote {zp}")
json.dump(result, open(os.path.join(OUT, f"{NAME}_result.json"), "w"), indent=1, default=str)
log(f"done: {NAME} | best dev macro-F1 {best:.4f} | train {train_min:.1f} min | peak {peak_gb:.1f} GB | "
    f"trainable {n_train/1e6:.3f}M of {n_total/1e6:.2f}M")

In [ ]:
%%writefile phase1_loader.py
"""phase1_loader.py — load a Phase-1 model with its trained weights."""
import importlib, json, os

import torch

REGISTRY = os.environ.get("PHASE1_REGISTRY", "phase1_registry.json")


def load_phase1(name, map_location="cpu"):
    reg = json.load(open(REGISTRY))
    assert name in reg, f"{name!r} not in {REGISTRY}: {sorted(reg)}"
    r = reg[name]
    mod = importlib.import_module(r["module"])
    cfg = dict(r["model_cfg"])
    model = mod.build_model(cfg)
    ck = torch.load(r["ckpt"], map_location=map_location, weights_only=False)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, f"{name}: missing {missing[:5]} unexpected {unexpected[:5]}"
    val = ck.get("val_loss") if isinstance(ck, dict) else None
    return model, dict(model_cfg=cfg, ckpt=r["ckpt"], module=r["module"], val_loss=r.get("val_loss", val))

In [ ]:
%%writefile cj_dummy.py
"""Smoke-test model only (NOT a submission model): bag of embeddings per chunk, mean over a case's chunks."""
import torch
import torch.nn as nn


class Dummy(nn.Module):
    def __init__(self, vocab_size, num_labels):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 32)
        self.out = nn.Linear(32, num_labels)

    def forward(self, input_ids, attention_mask, doc_ids, n_docs, chunk_pos=None, chunk_rel=None):
        m = attention_mask.float()[..., None]
        chunk = (self.emb(input_ids) * m).sum(1) / m.sum(1).clamp(min=1)
        doc = torch.zeros(n_docs, chunk.shape[1], device=chunk.device, dtype=chunk.dtype).index_add_(0, doc_ids, chunk)
        cnt = torch.zeros(n_docs, device=chunk.device, dtype=chunk.dtype).index_add_(0, doc_ids, torch.ones_like(doc_ids, dtype=chunk.dtype))
        return self.out(doc / cnt.clamp(min=1)[:, None])


def build_cj_model(model_cfg, num_labels):
    return Dummy(int(model_cfg["vocab_size"]), num_labels)

In [ ]:
%%writefile p1_biscale.py
"""model.py for the Bi-Scale generation notebook."""
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================== copied from the Bi-Scale training notebook ==============================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq=2048, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self._build_cache(max_seq)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device).float()
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, seq_len):
        if seq_len > self.cos_cached.shape[2]:
            self._build_cache(seq_len)
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]

def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rotary(x, cos, sin):
    return x * cos + rotate_half(x) * sin

class BiScaleGQA(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout, ctx_len, rope_theta, is_local=False, window_size=256):
        super().__init__()
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups   = n_heads // n_kv_heads
        self.d_head     = d_model // n_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.o_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.d_head, max_seq=ctx_len, theta=rope_theta)

        causal = torch.tril(torch.ones(ctx_len, ctx_len))
        if is_local:
            causal = causal * torch.triu(torch.ones(ctx_len, ctx_len), diagonal=-window_size)
        self.register_buffer("mask", causal.view(1, 1, ctx_len, ctx_len))


        for proj in [self.q_proj, self.k_proj, self.v_proj, self.o_proj]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        B, T, C = x.shape
        dh = self.d_head

        q = self.q_proj(x).view(B, T, self.n_heads, dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)

        cos, sin = self.rope(T)
        q = apply_rotary(q, cos, sin)
        k = apply_rotary(k, cos, sin)

        k = k.repeat_interleave(self.n_groups, dim=1)
        v = v.repeat_interleave(self.n_groups, dim=1)

        scale = dh ** -0.5
        att = (q @ k.transpose(-2, -1)) * scale
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = self.attn_drop(F.softmax(att, dim=-1))

        out = (att @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up   = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        for proj in [self.gate, self.up, self.down]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class LearnedGatedResidual(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Linear(dim, dim, bias=True)

        nn.init.normal_(self.gate.weight, mean=0.0, std=0.01)
        nn.init.constant_(self.gate.bias, 2.0)

    def forward(self, x, sublayer):
        g = torch.sigmoid(self.gate(x))
        return x + g * sublayer

class BiScaleLegalBlock(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        is_local = (layer_idx < (cfg["n_layers"] // 2))
        self.norm1 = RMSNorm(cfg["d_model"])
        self.attn  = BiScaleGQA(
            cfg["d_model"], cfg["n_heads"], cfg["n_kv_heads"],
            cfg["dropout"], cfg["ctx_len"], cfg["rope_theta"],
            is_local=is_local, window_size=cfg["local_window"]
        )
        self.res1 = LearnedGatedResidual(cfg["d_model"])
        self.norm2 = RMSNorm(cfg["d_model"])
        self.ffn   = SwiGLU(cfg["d_model"], cfg["d_ff"])
        self.res2  = LearnedGatedResidual(cfg["d_model"])
        self.drop  = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        x = self.res1(x, self.drop(self.attn(self.norm1(x))))
        x = self.res2(x, self.drop(self.ffn(self.norm2(x))))
        return x

class BiScaleLegalLLaMA(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.ctx     = cfg["ctx_len"]
        self.tok_emb = nn.Embedding(vocab_size, cfg["d_model"])
        self.drop    = nn.Dropout(cfg["dropout"])
        self.blocks  = nn.ModuleList([BiScaleLegalBlock(cfg, i) for i in range(cfg["n_layers"])])
        self.norm    = RMSNorm(cfg["d_model"])
        self.lm_head = nn.Linear(cfg["d_model"], vocab_size, bias=False)


        self.lm_head.weight = self.tok_emb.weight


        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=0.02)

    def forward(self, ids, targets=None):
        B, T = ids.shape
        x = self.drop(self.tok_emb(ids))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
# ======================================= end of copied code =======================================


_ORIGINAL_ATTN_FORWARD = BiScaleGQA.forward


def _sdpa_attn_forward(self, x):
    B, T, C = x.shape
    dh = self.d_head
    q = self.q_proj(x).view(B, T, self.n_heads, dh).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
    cos, sin = self.rope(T)
    q = apply_rotary(q, cos.to(q.dtype), sin.to(q.dtype))
    k = apply_rotary(k, cos.to(k.dtype), sin.to(k.dtype))
    k = k.repeat_interleave(self.n_groups, dim=1)
    v = v.repeat_interleave(self.n_groups, dim=1)
    y = F.scaled_dot_product_attention(q, k, v, attn_mask=self.mask[:, :, :T, :T].bool(),
                                       dropout_p=0.0, scale=dh ** -0.5)
    return self.o_proj(y.transpose(1, 2).contiguous().view(B, T, -1))


def use_sdpa(flag=True):
    BiScaleGQA.forward = _sdpa_attn_forward if flag else _ORIGINAL_ATTN_FORWARD


class BiScaleForHarness(BiScaleLegalLLaMA):
    def forward(self, input_ids, prefix_lens=None):
        return super().forward(input_ids)[0]


def build_model(model_cfg):
    use_sdpa(model_cfg.get("use_sdpa", True))
    return BiScaleForHarness(model_cfg, model_cfg["vocab_size"])

In [ ]:
# 3 — data check: layout, sizes, label balance, document lengths; builds the token cache all runs share
!python test_cj_data.py | tail -1
import cj_data as cd, importlib; importlib.reload(cd)
from tokenizers import ByteLevelBPETokenizer
_tok = ByteLevelBPETokenizer(vocab="tokenizer/vocab.json", merges="tokenizer/merges.txt")
V = _tok.get_vocab_size()
enc_batch = lambda texts: [e.ids for e in _tok.encode_batch(texts)]
CASES, TOKS = {}, {}
for split in ("train", "dev", "test"):
    raw = json.load(open(P4[split]))
    ex = raw[0] if isinstance(raw, list) else next(iter(raw.items()))
    print(f"{split}.json layout: {type(raw).__name__} | example: {str(ex)[:200]} ...")
    t0 = time.time()
    CASES[split] = cd.load_cases(P4[split])
    TOKS[split] = cd.tokenize_cases(CASES[split], enc_batch, "cj_cache", tag=split)
    print(f"  {split}: " + json.dumps(cd.dataset_stats(CASES[split], TOKS[split])) + f" | tokenized in {time.time()-t0:.0f}s")
ov = set(map(str, (v["id"] for v in CASES["test"].values()))) & set(map(str, (v["id"] for v in CASES["train"].values())))
print("test ids also in train:", len(ov))
for L in (512, 1024):
    n = sorted(-(-max(1, len(v)) // L) for v in TOKS["train"].values())
    print(f"chunks of {L} tokens per train case: p50 {n[len(n)//2]} | p90 {n[int(len(n)*.9)]} | max {n[-1]}")

In [ ]:
# 4 — Phase-1 models -> phase1_registry.json (same as Phase 2)
REG = {}
V = _tok.get_vocab_size()
def near(ckpt, fn):
    root = os.path.dirname(os.path.dirname(ckpt)) if os.path.basename(os.path.dirname(ckpt)) == "run" else os.path.dirname(ckpt)
    hits = sorted(glob.glob(f"{root}/**/{fn}", recursive=True), key=len)
    return hits[0] if hits else None
for name in ["prefixlm", "encdec", "arch4"]:
    ck = find(f"{name}_best.pt")
    if not ck:
        print(f"{name}: NOT FOUND (attach phase1-models)"); continue
    shutil.copy(near(ck[0], "model.py"), f"p1_{name}.py")
    cfgj = json.load(open(near(ck[0], "run_config.json")))
    mc = dict(cfgj["model"]); mc.setdefault("vocab_size", V); mc.setdefault("max_seq_len", int(cfgj["train"]["ctx_len"]))
    REG[name] = dict(module=f"p1_{name}", model_cfg=mc, ckpt=ck[0])
bs = find("stage2_done.pt")
if bs:
    REG["biscale"] = dict(module="p1_biscale", ckpt=bs[0],
                          model_cfg=dict(n_layers=16, n_heads=12, n_kv_heads=4, d_model=768, d_ff=2048, ctx_len=1024,
                                         local_window=256, dropout=0.1, rope_theta=10000.0, vocab_size=V))
json.dump(REG, open("phase1_registry.json", "w"), indent=1)
from phase1_loader import load_phase1
for name in REG:
    m, info = load_phase1(name)
    print(f"{name:9s} {sum(p.numel() for p in m.parameters())/1e6:7.2f}M params | {info['ckpt']}")
    del m

In [ ]:
# 5 — experiment config helper + parallel runner
TODO = "TODO"
R = "/kaggle/working/p4runs"
def make_cfg(run_name, model, train, data=None, kd=None, dump_teacher=False, module="cj_model", teacher_cases=None):
    d = dict(tokenizer_dir="tokenizer", train=P4["train"], dev=P4["dev"], test=P4["test"], cache_dir="cj_cache",
             chunk_len=TODO,
             stride=None,
             max_chunks=TODO,
             select=TODO)

    d.update(data or {})
    cfg = dict(run_name=run_name, out_dir=f"{R}/{run_name}", model_module=module, data=d, model=model, train=train,
               kd=kd, dump_teacher_logits=dump_teacher, teacher_cases=teacher_cases)
    todo = [k for k, v in {**d, **train, **(kd or {}), **model}.items() if v == TODO]
    assert not todo, f"{run_name}: fill {todo}"
    os.makedirs(cfg["out_dir"], exist_ok=True)
    p = f"{cfg['out_dir']}/config.json"; json.dump(cfg, open(p, "w"), indent=1)
    return p

def run_parallel(cfg_paths, timeout_min=240):
    procs, queue = [], list(cfg_paths)
    while queue or procs:
        while queue and len(procs) < max(1, NGPU):
            cp = queue.pop(0); gpu = [g for g in range(max(1, NGPU)) if g not in [x[2] for x in procs]][0]
            lf = open(os.path.join(os.path.dirname(cp), "train.log"), "w")
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
            procs.append((subprocess.Popen([sys.executable, "cj_train.py", cp], stdout=lf, stderr=subprocess.STDOUT, env=env,
                                           start_new_session=True), cp, gpu, lf, time.time()))
            print(f"started {cp} on GPU {gpu}")
        time.sleep(15)
        for pr in list(procs):
            p, cp, gpu, lf, t0 = pr
            if p.poll() is not None or time.time() - t0 > timeout_min * 60:
                if p.poll() is None: p.kill()
                lf.close(); procs.remove(pr)
                tail = open(os.path.join(os.path.dirname(cp), "train.log")).read().splitlines()[-25:]
                print(f"\n==== {cp} exit {p.returncode} ({(time.time()-t0)/60:.1f} min)\n" + "\n".join(tail))

def summary(names):
    for n in names:
        z, res = f"{R}/{n}/{n}_submission.zip", f"{R}/{n}/{n}_result.json"
        if os.path.exists(z) and os.path.exists(res):
            r = json.load(open(res)); dv = r.get("dev", {})
            print(f"OK  {n:22s} dev macro-F1 {dv.get('macro_f1', float('nan')):.4f} (argmax {r.get('dev_argmax', {}).get('macro_f1', float('nan')):.4f}, "
                  f"threshold {r.get('threshold')}) | test preds {r.get('test_pred_counts')} | trainable {r['params_trainable']/1e6:.2f}M of "
                  f"{r['params_total']/1e6:.1f}M | {r['train_minutes']:.1f} min -> {z}")
        else:
            print(f"FAILED {n}: read {R}/{n}/train.log")

In [ ]:
# 6 — smoke test: dummy model, two chunking setups, dev-tuned threshold, teacher dump -> KD student, zip
SM_T = dict(epochs=2, epoch_cases=256, lr=1e-3, batch_size=16, max_minutes=2)
a = make_cfg("smoke_teacher", dict(vocab_size=V), SM_T, dict(chunk_len=512, max_chunks=2, select="head_tail"),
             module="cj_dummy", dump_teacher=True, teacher_cases=500)
b = make_cfg("smoke_thresh", dict(vocab_size=V), dict(SM_T, threshold="dev_tuned", class_weights="balanced"),
             dict(chunk_len=256, stride=192, max_chunks=3, select="last", eval_max_chunks=4), module="cj_dummy")
run_parallel([a, b])
c = make_cfg("smoke_kd", dict(vocab_size=V), SM_T, dict(chunk_len=512, max_chunks=2, select="head_tail"), module="cj_dummy",
             kd=dict(teacher_logits=f"{R}/smoke_teacher/smoke_teacher_teacher_logits.npz", temperature=2.0, alpha=0.5))
run_parallel([c])
summary(["smoke_teacher", "smoke_thresh", "smoke_kd"])
assert all(os.path.exists(f"{R}/{n}/{n}_submission.zip") for n in ["smoke_teacher", "smoke_thresh", "smoke_kd"]), "SMOKE TEST FAILED"
import zipfile; zf = zipfile.ZipFile(f"{R}/smoke_kd/smoke_kd_submission.zip")
print("zip contains:", zf.namelist(), "| first entries:", json.loads(zf.read("prediction.json"))[:3])
print("SMOKE TEST PASSED (pipeline only: the dummy model's scores mean nothing)")

In [ ]:
%%writefile cj_model.py
"""cj_model.py — Phase-4 model."""
import torch
import torch.nn as nn

from phase1_loader import load_phase1


def build_cj_model(model_cfg: dict, num_labels: int) -> nn.Module:
    raise NotImplementedError("write your Phase-4 model here")

In [ ]:
# 8 — model check: output shape [n_docs, 2], trainable params, gradients reach every trainable tensor, memory
import importlib, cj_model; importlib.reload(cj_model)
CHECK_MODEL = dict(TODO="TODO")
m = cj_model.build_cj_model(dict(CHECK_MODEL), 2).cuda().train()
n_docs, per_doc, T = 4, 3, 512
N = n_docs * per_doc
x = torch.randint(5, V, (N, T), device="cuda"); msk = torch.ones_like(x, dtype=torch.bool); msk[-1, T // 2:] = False
did = torch.arange(n_docs, device="cuda").repeat_interleave(per_doc)
pos = torch.arange(per_doc, device="cuda").repeat(n_docs); rel = pos.float() / per_doc
torch.cuda.reset_peak_memory_stats()
with torch.autocast("cuda", dtype=torch.float16):
    out = m(x, msk, did, n_docs, chunk_pos=pos, chunk_rel=rel)
assert tuple(out.shape) == (n_docs, 2), f"logits must be [n_docs, 2], got {tuple(out.shape)}"
out.float().mean().backward()
nog = [n for n, p in m.named_parameters() if p.requires_grad and p.grad is None]
tot, tr = sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"total {tot/1e6:.2f}M | trainable {tr/1e6:.3f}M ({100*tr/tot:.2f}%) | peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB"
      f" | trainable but no gradient: {nog[:10]}")
del m, out; torch.cuda.empty_cache()

In [ ]:
# 9 — EXPERIMENTS (YOURS): make_cfg(...) per experiment, run_parallel([...]) two at a time, then summary([...])

In [ ]:
# 10 — results table (report) + download links + small backup zip (no weights)
import pandas as pd
from IPython.display import FileLink, display
rows = []
for rp in sorted(glob.glob(f"{R}/*/*_result.json")):
    r = json.load(open(rp))
    if r["run"].startswith("smoke"): continue
    dv = r.get("dev", {})
    rows.append(dict(run=r["run"], dev_macro_f1=dv.get("macro_f1"), dev_argmax_f1=r.get("dev_argmax", {}).get("macro_f1"),
                     threshold=r.get("threshold"), dev_acc=dv.get("accuracy"), trainable_M=r["params_trainable"]/1e6,
                     total_M=r["params_total"]/1e6, train_min=r["train_minutes"], peak_GB=r["peak_mem_gb"],
                     epochs_done=len(r.get("history", [])), chunk_len=r["chunk_len"], max_chunks=r["max_chunks"], select=r["select"]))
df = pd.DataFrame(rows).sort_values("run"); display(df)
df.to_csv(f"{R}/results.csv", index=False)
os.chdir("/kaggle/working")
for n in df.run: display(FileLink(f"p4runs/{n}/{n}_submission.zip"))
!zip -qr p4_backup.zip cj_model.py p4runs -x "*.pt" "*.npz" "*smoke*"
display(FileLink("p4_backup.zip"))